# 📖 Notebook 3: Retention Policies & Downsampling

Storing 30-second resolution data forever is expensive and pointless — nobody needs millisecond precision for last year's CPU data. This notebook shows how to:

1. **Automatically delete old data** with retention policies
2. **Pre-compute rollups** with continuous aggregates (downsampling)
3. Layer both together for a production-grade data lifecycle

## Learning Objectives

By the end of this notebook you will understand:
- How TimescaleDB retention policies drop old chunks automatically
- How continuous aggregates pre-compute time-bucketed summaries in the background
- A real-world tiered storage strategy: raw → 5-min → 1-hour rollups
- The storage savings from downsampling

## 🛠️ Setup

```bash
cd deep-dives/time-series-databases
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "tsdb_demo",
    "user": "demo",
    "password": "demo"
}

def run_query(sql, params=None):
    with psycopg2.connect(**DB_CONFIG) as conn:
        return pd.read_sql_query(sql, conn, params=params)

def run_exec(sql):
    with psycopg2.connect(**DB_CONFIG) as conn:
        conn.autocommit = True
        with conn.cursor() as cur:
            cur.execute(sql)

print("✅ Connected!")

---
## 1 — Current Storage Baseline

Before we change anything, let's see how much space our raw data uses.

In [ ]:
size = run_query("""
    SELECT
        hypertable_name,
        pg_size_pretty(hypertable_size('metrics')) AS total_size,
        num_chunks
    FROM timescaledb_information.hypertables
    WHERE hypertable_name = 'metrics'
""")
size

In [ ]:
# Size per chunk (partition)
chunks = run_query("""
    SELECT
        chunk_name,
        range_start,
        range_end,
        pg_size_pretty(total_bytes) AS size
    FROM timescaledb_information.chunks
    WHERE hypertable_name = 'metrics'
    ORDER BY range_start
""")
print(f"Chunks: {len(chunks)}")
chunks

---
## 2 — Retention Policies: Auto-Delete Old Data

A **retention policy** tells TimescaleDB: *"Automatically drop chunks older than X."*

Because data is already partitioned by time, dropping a chunk is like deleting a file — instant, no expensive row-by-row `DELETE`.  
This is the *"retention becomes trivial"* insight from the article.

Let's add a 5-day retention policy on raw data.

> **Note**: In production you'd keep raw data for 7-30 days. We use 5 days here so we can see the effect immediately with our 7-day seed data.

In [ ]:
# Count rows BEFORE retention
before = run_query("SELECT count(*) AS n FROM metrics")
print(f"Rows before retention: {before['n'].iloc[0]:,}")

# Add a retention policy: drop chunks older than 5 days
run_exec("""
    SELECT add_retention_policy('metrics', drop_after => INTERVAL '5 days', if_not_exists => true)
""")
print("✅ Retention policy added (drop chunks older than 5 days)")

In [ ]:
# The policy runs on a schedule. Let's trigger it manually to see the effect now.
run_exec("""
    CALL run_job((SELECT job_id FROM timescaledb_information.jobs WHERE proc_name = 'policy_retention' LIMIT 1))
""")

after = run_query("SELECT count(*) AS n FROM metrics")
print(f"Rows after retention:  {after['n'].iloc[0]:,}")
print(f"Rows dropped:          {before['n'].iloc[0] - after['n'].iloc[0]:,}")

In [ ]:
# Check which chunks survived
remaining = run_query("""
    SELECT chunk_name, range_start, range_end, pg_size_pretty(total_bytes) AS size
    FROM timescaledb_information.chunks
    WHERE hypertable_name = 'metrics'
    ORDER BY range_start
""")
print(f"Chunks remaining: {len(remaining)} (older chunks were dropped)")
remaining

Dropping a chunk is **O(1)** — it just removes the underlying file. Compare this to a `DELETE FROM metrics WHERE time < X` on a regular table, which would need to scan and remove rows one by one.

---
## 3 — Continuous Aggregates: Pre-Computed Rollups

A **continuous aggregate** is a materialized view that TimescaleDB keeps up-to-date automatically.  
It pre-computes `time_bucket` + aggregation so that dashboards read from the rollup instead of scanning raw data.

This is the **downsampling** concept from the article:
- Raw data (30s) → **5-minute rollup** → **1-hour rollup**
- Each level stores min, max, avg, count so you can answer most queries without touching raw data.

### Create a 5-minute rollup

In [ ]:
run_exec("""
    CREATE MATERIALIZED VIEW IF NOT EXISTS metrics_5min
    WITH (timescaledb.continuous) AS
    SELECT
        time_bucket('5 minutes', time) AS bucket,
        host,
        region,
        metric_name,
        avg(value)   AS avg_value,
        min(value)   AS min_value,
        max(value)   AS max_value,
        count(*)     AS sample_count
    FROM metrics
    GROUP BY bucket, host, region, metric_name
    WITH NO DATA
""")
print("✅ Created continuous aggregate: metrics_5min")

In [ ]:
# Manually refresh to backfill historical data
run_exec("""
    CALL refresh_continuous_aggregate('metrics_5min', now() - interval '7 days', now())
""")
print("✅ Backfilled metrics_5min")

run_query("SELECT count(*) AS rows FROM metrics_5min")

In [ ]:
# Set up an auto-refresh policy so new data is rolled up automatically
run_exec("""
    SELECT add_continuous_aggregate_policy('metrics_5min',
        start_offset  => INTERVAL '1 hour',
        end_offset    => INTERVAL '5 minutes',
        schedule_interval => INTERVAL '5 minutes',
        if_not_exists => true
    )
""")
print("✅ Auto-refresh policy added — metrics_5min will stay up to date")

### Create a 1-hour rollup

We can even build a rollup **on top of another rollup** for very long-term queries.

In [ ]:
run_exec("""
    CREATE MATERIALIZED VIEW IF NOT EXISTS metrics_1hour
    WITH (timescaledb.continuous) AS
    SELECT
        time_bucket('1 hour', bucket) AS bucket,
        host,
        region,
        metric_name,
        avg(avg_value)   AS avg_value,
        min(min_value)   AS min_value,
        max(max_value)   AS max_value,
        sum(sample_count) AS sample_count
    FROM metrics_5min
    GROUP BY bucket, host, region, metric_name
    WITH NO DATA
""")
print("✅ Created continuous aggregate: metrics_1hour")

run_exec("""
    CALL refresh_continuous_aggregate('metrics_1hour', now() - interval '7 days', now())
""")
print("✅ Backfilled metrics_1hour")

run_exec("""
    SELECT add_continuous_aggregate_policy('metrics_1hour',
        start_offset  => INTERVAL '3 hours',
        end_offset    => INTERVAL '1 hour',
        schedule_interval => INTERVAL '1 hour',
        if_not_exists => true
    )
""")
print("✅ Auto-refresh policy added for metrics_1hour")

---
## 4 — Querying Rollups vs. Raw Data

Now we have three tiers:

| Tier | Resolution | Best For |
|------|-----------|----------|
| `metrics` | 30 seconds | Debugging recent issues |
| `metrics_5min` | 5 minutes | Hourly/daily dashboards |
| `metrics_1hour` | 1 hour | Weekly/monthly trends |

Let's compare query speed across all three.

In [ ]:
import time as _time

queries = {
    "raw (metrics)": """
        SELECT date_trunc('hour', time) AS hour, avg(value)
        FROM metrics
        WHERE host = 'server-1' AND metric_name = 'cpu_usage'
          AND time > now() - interval '3 days'
        GROUP BY hour ORDER BY hour
    """,
    "5-min rollup": """
        SELECT date_trunc('hour', bucket) AS hour, avg(avg_value)
        FROM metrics_5min
        WHERE host = 'server-1' AND metric_name = 'cpu_usage'
          AND bucket > now() - interval '3 days'
        GROUP BY hour ORDER BY hour
    """,
    "1-hour rollup": """
        SELECT bucket AS hour, avg_value
        FROM metrics_1hour
        WHERE host = 'server-1' AND metric_name = 'cpu_usage'
          AND bucket > now() - interval '3 days'
        ORDER BY hour
    """
}

for label, sql in queries.items():
    start = _time.perf_counter()
    df = run_query(sql)
    elapsed = (_time.perf_counter() - start) * 1000
    print(f"{label:20s} → {elapsed:6.1f} ms  ({len(df)} rows)")

The rollup queries are faster because they scan **far fewer rows** — the aggregation is already done.

---
## 5 — Storage Comparison

Let's see how much space each tier uses.

In [ ]:
storage = run_query("""
    SELECT
        view_name AS name,
        pg_size_pretty(materialization_hypertable_size) AS size
    FROM timescaledb_information.continuous_aggregates
""")

raw_size = run_query("SELECT pg_size_pretty(hypertable_size('metrics')) AS size")
print(f"{'metrics (raw)':25s} : {raw_size['size'].iloc[0]}")
for _, row in storage.iterrows():
    print(f"{row['name']:25s} : {row['size']}")

In [ ]:
# Row counts across tiers
for table in ['metrics', 'metrics_5min', 'metrics_1hour']:
    count_col = 'bucket' if table != 'metrics' else 'time'
    n = run_query(f"SELECT count(*) AS n FROM {table}")
    print(f"{table:20s} : {n['n'].iloc[0]:>10,} rows")

The 1-hour rollup is **tiny** compared to raw data but can still answer most long-range dashboard queries.

---
## 6 — The Full Data Lifecycle

Here is a production-grade tiered strategy:

```
┌─────────────────────────────────────────────────────────────┐
│              DATA LIFECYCLE                                  │
├──────────────┬──────────────┬───────────────────────────────┤
│ Age          │ Tier         │ Policy                        │
├──────────────┼──────────────┼───────────────────────────────┤
│ 0 – 7 days  │ Raw (30s)    │ Full resolution               │
│ 0 – 30 days │ 5-min rollup │ Continuous aggregate          │
│ 0 – 1 year  │ 1-hour rollup│ Continuous aggregate          │
├──────────────┼──────────────┼───────────────────────────────┤
│ > 7 days     │ Raw          │ Retention policy → DROP       │
│ > 30 days    │ 5-min rollup │ Retention policy → DROP       │
│ > 1 year     │ 1-hour rollup│ Retention policy → DROP       │
└──────────────┴──────────────┴───────────────────────────────┘
```

The key insight: **you never lose the ability to answer queries** about historical data — you just answer them at lower resolution.  
"What was average CPU last Tuesday?" → answered by the 5-min rollup.  
"What was average CPU in January?" → answered by the 1-hour rollup.

---
## 7 — Visualize All Three Tiers

Let's plot the same time range from all three tiers to see the resolution difference.

In [ ]:
raw = run_query("""
    SELECT time AS ts, value AS val FROM metrics
    WHERE host = 'server-1' AND metric_name = 'cpu_usage'
      AND time > now() - interval '24 hours'
    ORDER BY ts
""")

five = run_query("""
    SELECT bucket AS ts, avg_value AS val FROM metrics_5min
    WHERE host = 'server-1' AND metric_name = 'cpu_usage'
      AND bucket > now() - interval '24 hours'
    ORDER BY ts
""")

hour = run_query("""
    SELECT bucket AS ts, avg_value AS val FROM metrics_1hour
    WHERE host = 'server-1' AND metric_name = 'cpu_usage'
      AND bucket > now() - interval '24 hours'
    ORDER BY ts
""")

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(raw['ts'], raw['val'], linewidth=0.4, alpha=0.8)
axes[0].set_title(f'Raw 30s data ({len(raw)} points)')
axes[0].set_ylabel('CPU %')

axes[1].plot(five['ts'], five['val'], linewidth=1, color='orange')
axes[1].set_title(f'5-min rollup ({len(five)} points)')
axes[1].set_ylabel('CPU %')

axes[2].step(hour['ts'], hour['val'], linewidth=1.5, color='green', where='mid')
axes[2].set_title(f'1-hour rollup ({len(hour)} points)')
axes[2].set_ylabel('CPU %')
axes[2].set_xlabel('Time')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('Same data at three resolutions — server-1 CPU (24h)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8 — Cleanup: View Active Policies

Let's see all the automated jobs TimescaleDB is now running for us.

In [ ]:
jobs = run_query("""
    SELECT
        job_id,
        proc_name,
        hypertable_name,
        schedule_interval,
        next_start
    FROM timescaledb_information.jobs
    WHERE hypertable_name IS NOT NULL
    ORDER BY job_id
""")
jobs

---
## 🧠 Key Takeaways

1. **Retention policies** auto-drop old chunks — instant and free compared to row-level `DELETE`.
2. **Continuous aggregates** are materialized views that TimescaleDB refreshes automatically — they pre-compute `time_bucket` + aggregation.
3. **Tiered storage** (raw → 5-min → 1-hour) trades precision for efficiency on older data.
4. Rollups preserve `min`, `max`, `avg`, `count` so you can still answer most analytical queries.
5. This pattern is universal in monitoring systems: Prometheus, Datadog, Grafana Cloud all do the same thing.

## 🎓 Congratulations!

You've completed the Time-Series Databases lab series. You now understand:
- How time-series data is modeled (Notebook 1)
- How to aggregate and analyze it with windowed queries (Notebook 2)
- How to manage its lifecycle with retention and downsampling (Notebook 3)

These are the same building blocks behind systems like Prometheus, InfluxDB, and Datadog.